In [1]:
import os
import sys

!git clone https://github.com/RuoyuLi97/info7160-goemotions-emotion-classification.git
os.chdir("info7160-goemotions-emotion-classification")
sys.path.append(os.getcwd())

fatal: destination path 'info7160-goemotions-emotion-classification' already exists and is not an empty directory.


In [2]:
import os
import sys

os.chdir("info7160-goemotions-emotion-classification")
!git pull origin main
sys.path.append(os.getcwd())

From https://github.com/RuoyuLi97/info7160-goemotions-emotion-classification
 * branch            main       -> FETCH_HEAD
Already up to date.


In [3]:
!pip install datasets transformers torch scikit-learn numpy -q

import json
import numpy as np
import pandas as pd
from src.preprocessing import LABEL_NAMES
from src.metrics import compute_metrics, compute_per_label_f1

In [4]:
def load_jsonl(path):
    data = []
    with open(path, "r") as f:
        for line in f:
            data.append(json.loads(line))

    df = pd.DataFrame(data)

    # detect if labels are binary vectors or string lists
    sample = df["true_labels"].iloc[0]
    if isinstance(sample, list) and isinstance(sample[0], (int, float)):
        # binary vector format — convert to string labels
        df["true_labels"] = df["true_labels"].apply(
            lambda vec: [LABEL_NAMES[i] for i, v in enumerate(vec) if v == 1]
        )
        df["pred_labels"] = df["pred_labels"].apply(
            lambda vec: [LABEL_NAMES[i] for i, v in enumerate(vec) if v == 1]
        )

    return df

df_base = load_jsonl("outputs/baseline_preds.jsonl")
df_bert = load_jsonl("outputs/bert_preds.jsonl")

print(f"Baseline examples: {len(df_base)}")
print(f"BERT examples: {len(df_bert)}")
print(f"\nBaseline sample:")
print(df_base.head(2))
print(f"\nBERT sample:")
print(df_bert.head(2))

Baseline examples: 5427
BERT examples: 5427

Baseline sample:
        id                                               text   true_labels  \
0  eecwqtt  I’m really sorry about your situation :( Altho...     [sadness]   
1  ed5f85d    It's wonderful because it's awful. At not with.  [admiration]   

       pred_labels  
0  [love, remorse]  
1        [disgust]  

BERT sample:
        id                                               text   true_labels  \
0  eecwqtt  I’m really sorry about your situation :( Altho...     [sadness]   
1  ed5f85d    It's wonderful because it's awful. At not with.  [admiration]   

    pred_labels  
0        [love]  
1  [admiration]  


In [5]:
label_to_idx = {l: i for i, l in enumerate(LABEL_NAMES)}

def to_multi_hot(label_lists):
    mat = np.zeros((len(label_lists), len(LABEL_NAMES)), dtype=np.float32)
    for i, labels in enumerate(label_lists):
        for l in labels:
            if l in label_to_idx:
                mat[i][label_to_idx[l]] = 1
    return mat

y_true_base = to_multi_hot(df_base["true_labels"])
y_pred_base = to_multi_hot(df_base["pred_labels"])

y_true_bert = to_multi_hot(df_bert["true_labels"])
y_pred_bert = to_multi_hot(df_bert["pred_labels"])

print(f"True labels matrix shape: {y_true_base.shape}")
print(f"Pred labels matrix shape: {y_pred_base.shape}")

True labels matrix shape: (5427, 28)
Pred labels matrix shape: (5427, 28)


In [6]:
metrics_base = compute_metrics(y_true_base, y_pred_base)
metrics_bert = compute_metrics(y_true_bert, y_pred_bert)

comparison = pd.DataFrame({
    "Metric": list(metrics_base.keys()),
    "Baseline": list(metrics_base.values()),
    "BERT": list(metrics_bert.values())
})

print(comparison.to_string(index=False))

   Metric  Baseline   BERT
 micro_f1    0.5287 0.5586
 macro_f1    0.3268 0.3025
precision    0.5899 0.5880
   recall    0.4791 0.5320


In [7]:
per_label_base = compute_per_label_f1(y_true_base, y_pred_base, LABEL_NAMES)
per_label_bert = compute_per_label_f1(y_true_bert, y_pred_bert, LABEL_NAMES)

label_df = pd.DataFrame({
    "label": LABEL_NAMES,
    "baseline_f1": list(per_label_base.values()),
    "bert_f1": list(per_label_bert.values())
})

print("Top 5 easiest labels (by BERT F1):")
print(label_df.sort_values("bert_f1", ascending=False).head(5).to_string(index=False))

print("\nTop 5 hardest labels (by BERT F1):")
print(label_df.sort_values("bert_f1").head(5).to_string(index=False))

Top 5 easiest labels (by BERT F1):
     label  baseline_f1  bert_f1
 gratitude       0.9142   0.9162
 amusement       0.7343   0.8163
      love       0.7328   0.8068
   neutral       0.6469   0.6683
admiration       0.5950   0.6648

Top 5 hardest labels (by BERT F1):
        label  baseline_f1  bert_f1
       caring       0.2222      0.0
         fear       0.4466      0.0
embarrassment       0.1463      0.0
       desire       0.2545      0.0
        pride       0.1111      0.0


In [8]:
def is_wrong(true, pred):
    return set(true) != set(pred)

errors = df_bert[
    df_bert.apply(
        lambda row: is_wrong(row["true_labels"], row["pred_labels"]),
        axis=1
    )
].head(8)

for _, row in errors.iterrows():
    print(f"Text: {row['text'][:100]}")
    print(f"True:      {row['true_labels']}")
    print(f"Predicted: {row['pred_labels']}")
    print()

Text: I’m really sorry about your situation :( Although I love the names Sapphira, Cirilla, and Scarlett!
True:      ['sadness']
Predicted: ['love']

Text: Kings fan here, good luck to you guys! Will be an interesting game to watch! 
True:      ['excitement']
Predicted: ['optimism']

Text: I’m sorry to hear that friend :(. It’s for the best most likely if she didn’t accept you for who you
True:      ['remorse']
Predicted: ['remorse', 'sadness']

Text: Girlfriend weak as well, that jump was pathetic.
True:      ['sadness']
Predicted: ['anger', 'annoyance']

Text: [NAME] has towed the line of the Dark Side. He wouldn't cross it by doing something like this.
True:      ['annoyance', 'disapproval']
Predicted: ['neutral']

Text: Translation }}} I wish I could afford it.
True:      ['desire']
Predicted: []

Text: It's great that you're a recovering addict, that's cool. Have you ever tried DMT?
True:      ['admiration', 'curiosity']
Predicted: ['admiration']

Text: I've also heard that intrig

In [9]:
# save comparison table
comparison.to_csv("outputs/metrics_comparison.csv", index=False)

# save per-label breakdown
label_df.to_csv("outputs/per_label_f1.csv", index=False)

# save error examples
errors[["text", "true_labels", "pred_labels"]].to_csv("outputs/error_analysis.csv", index=False)

print("All results saved to outputs/")

All results saved to outputs/


In [14]:
# ── COVER PAGE ──────────────────────────────────────
TEAM_MEMBERS = [
    "Ruoyu Li",
    "Jinru Zhang",
    "Rongnan He",
    "Chengcheng Gou",
]
DATE   = "April 12, 2026"
COURSE = "INFO 7160 - Special Topics in Natural Language Engineering Methods and Tools"
# ─────────────────────────────────────────────────────────────

!pip install reportlab -q

from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.lib import colors
from reportlab.platypus import (SimpleDocTemplate, Paragraph, Spacer,
                                Table, TableStyle, PageBreak, HRFlowable)
from reportlab.lib.enums import TA_CENTER, TA_JUSTIFY, TA_LEFT
import os

PDF_NAME = "/content/GroupProject_GoEmotions_EmotionClassification_Report.pdf"
PAGE_W, PAGE_H = A4
MARGIN = 1.2 * cm
styles = getSampleStyleSheet()

def style(name, **kw):
    return ParagraphStyle(name, parent=styles["Normal"], **kw)

def tbl(data, col_widths, header=True):
    t = Table(data, colWidths=col_widths)
    ts = [
        ("BACKGROUND",    (0, 0), (-1, 0),  colors.HexColor("#f0f0f0")),
        ("FONTNAME",      (0, 0), (-1, 0),  "Helvetica-Bold"),
        ("FONTSIZE",      (0, 0), (-1, -1), 9),
        ("GRID",          (0, 0), (-1, -1), 0.4, colors.HexColor("#cccccc")),
        ("ROWBACKGROUNDS",(0, 1), (-1, -1),
         [colors.white, colors.HexColor("#fafafa")]),
        ("VALIGN",        (0, 0), (-1, -1), "MIDDLE"),
        ("LEFTPADDING",   (0, 0), (-1, -1), 6),
        ("RIGHTPADDING",  (0, 0), (-1, -1), 6),
        ("TOPPADDING",    (0, 0), (-1, -1), 4),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 4),
    ]
    t.setStyle(TableStyle(ts))
    return t

title_style    = style("T",  fontSize=20, leading=26, alignment=TA_CENTER,
                       fontName="Helvetica-Bold", textColor=colors.black, spaceAfter=6)
subtitle_style = style("ST", fontSize=12, leading=16, alignment=TA_CENTER,
                       fontName="Helvetica", textColor=colors.HexColor("#444444"), spaceAfter=4)
meta_style     = style("M",  fontSize=10, leading=14, alignment=TA_LEFT,
                       fontName="Helvetica", textColor=colors.black, spaceAfter=4)
h1_style       = style("H1", fontSize=14, leading=18, fontName="Helvetica-Bold",
                       textColor=colors.black, spaceBefore=14, spaceAfter=6)
h2_style       = style("H2", fontSize=11, leading=15, fontName="Helvetica-Bold",
                       textColor=colors.HexColor("#333333"), spaceBefore=10, spaceAfter=4)
body_style     = style("B",  fontSize=10, leading=14, alignment=TA_JUSTIFY,
                       fontName="Helvetica", textColor=colors.black, spaceAfter=6)
bullet_style   = style("BL", fontSize=10, leading=14, fontName="Helvetica",
                       textColor=colors.black, spaceAfter=4, leftIndent=16, bulletIndent=6)

doc = SimpleDocTemplate(
    PDF_NAME, pagesize=A4,
    leftMargin=MARGIN*2, rightMargin=MARGIN*2,
    topMargin=MARGIN*2, bottomMargin=MARGIN*2
)
story = []

# ── Cover Page (standalone) ──────────────────────────────────
story += [
    Spacer(1, 8*cm),
    Paragraph("Fine-grained Emotion Classification on GoEmotions", title_style),
    Paragraph("Baseline vs Transformer Models — A Comparative Study in Multi-label Emotion Recognition", subtitle_style),
    Spacer(1, 0.4*cm),
    Spacer(1, 0.4*cm),
    Paragraph(f"<b>Date:</b> {DATE}", meta_style),
    Paragraph(f"<b>Course:</b> {COURSE}", meta_style),
    Spacer(1, 0.2*cm),
    Paragraph("<b>Team Members:</b>", meta_style),
]
for member in TEAM_MEMBERS:
    story.append(Paragraph(f"&nbsp;&nbsp;&nbsp;&nbsp;{member}", meta_style))
story += [
    Spacer(1, 0.4*cm),
    PageBreak(),
]

# ── 1. Introduction ──────────────────────────────────────────
story += [
    Paragraph("1. Introduction", h1_style),
    Paragraph(
        "Traditional sentiment analysis classifies text into broad categories such as positive, "
        "negative, or neutral. However, human emotions are far more nuanced — a single sentence "
        "may simultaneously express admiration, curiosity, or disappointment rather than a single "
        "polarity. This project addresses fine-grained emotion classification, a multi-label task "
        "that assigns one or more of 28 emotion categories to each input text.",
        body_style),
    Paragraph(
        "We compare two approaches: a TF-IDF + Logistic Regression baseline representing "
        "traditional NLP methods, and a BERT-based transformer representing modern deep learning. "
        "Both models are evaluated on the GoEmotions benchmark dataset under identical conditions "
        "to enable a fair and meaningful comparison. The goal is not only to measure which model "
        "performs better, but to understand why — particularly in the context of class imbalance, "
        "threshold sensitivity, and training data constraints.",
        body_style),
]

# ── 2. Dataset ───────────────────────────────────────────────
story += [
    Paragraph("2. Dataset", h1_style),
    Paragraph(
        "We use the GoEmotions dataset released by Google Research (Demszky et al., 2020), "
        "available via HuggingFace Datasets. It consists of 54,263 English Reddit comments "
        "annotated with 28 emotion labels — 27 fine-grained emotions plus neutral — in a "
        "multi-label format where a single comment may carry more than one label.",
        body_style),
    tbl(
        [
            ["Split", "Examples", "Multi-label %", "Labels"],
            ["Train",      "43,410", "16.4%", "28"],
            ["Validation", "5,426",  "16.4%", "28"],
            ["Test",       "5,427",  "16.4%", "28"],
            ["Total",      "54,263", "16.4%", "28"],
        ],
        [5*cm, 4*cm, 4*cm, 3*cm]
    ),
    Spacer(1, 0.3*cm),
    Paragraph(
        "16.4% of training examples carry more than one label, confirming this is a genuine "
        "multi-label classification task rather than a single-label problem. The dataset "
        "exhibits severe class imbalance: neutral appears 14,219 times while grief appears "
        "only 77 times — a ratio of 184x. This imbalance directly impacts per-label model "
        "performance, especially for rare emotion categories.",
        body_style),
    tbl(
        [
            ["Top 5 Most Frequent", "Count", "Bottom 5 Least Frequent", "Count"],
            ["neutral",    "14,219", "embarrassment", "303"],
            ["admiration", "4,130",  "nervousness",   "164"],
            ["approval",   "2,939",  "relief",        "153"],
            ["gratitude",  "2,662",  "pride",         "111"],
            ["annoyance",  "2,470",  "grief",         "77"],
        ],
        [5*cm, 3*cm, 5*cm, 3*cm]
    ),
    Spacer(1, 0.3*cm),
    Paragraph(
        "The dataset is clean with no missing texts or empty label lists across all three splits, "
        "requiring no additional filtering or imputation during preprocessing.",
        body_style),
]

# ── 3. Approach ──────────────────────────────────────────────
story += [
    Paragraph("3. Approach", h1_style),
    Paragraph("3.1 Shared Preprocessing", h2_style),
    Paragraph(
        "A shared preprocessing module was developed for use by both models. Labels stored as "
        "integer indices are converted into 28-dimensional binary vectors using a multi-hot "
        "binarizer. For the transformer model, text is tokenized using BERT's WordPiece tokenizer "
        "with padding and truncation to a maximum sequence length of 128 tokens "
        "(Devlin et al., 2019). Evaluation metrics — micro F1, macro F1, precision, and recall "
        "— are computed using scikit-learn (Pedregosa et al., 2011).",
        body_style),
    Paragraph("3.2 Baseline — TF-IDF + Logistic Regression", h2_style),
    Paragraph(
        "The baseline converts raw text into TF-IDF feature vectors using a vocabulary of up "
        "to 50,000 terms with sublinear TF scaling. A MultiOutputClassifier wrapping Logistic "
        "Regression (max_iter=1000, lbfgs solver, C=1.0) is trained independently for each of "
        "the 28 labels on the full training set of 43,410 examples. A threshold sweep over "
        "{0.3, 0.4, 0.5} on the validation set identified 0.3 as the optimal decision threshold "
        "by micro F1 (validation micro F1 = 0.5257).",
        body_style),
    tbl(
        [
            ["Threshold", "Micro F1", "Macro F1", "Precision", "Recall"],
            ["0.3", "0.5257", "0.3315", "0.5942", "0.4713"],
            ["0.4", "0.4868", "0.2864", "0.6574", "0.3865"],
            ["0.5", "0.4153", "0.2439", "0.7114", "0.2933"],
        ],
        [3*cm, 3*cm, 3*cm, 3*cm, 3*cm]
    ),
    Spacer(1, 0.3*cm),
    Paragraph(
        "Lower thresholds recover more true labels for rare emotion categories at the cost "
        "of some precision, which is the expected tradeoff under severe class imbalance.",
        body_style),
    Paragraph("3.3 Transformer — BERT Fine-tuning", h2_style),
    Paragraph(
        "The transformer model uses bert-base-uncased (Devlin et al., 2019) loaded via "
        "HuggingFace Transformers (Wolf et al., 2020) with a linear classification head "
        "and sigmoid activation for 28-label multi-label output. Training uses "
        "BCEWithLogitsLoss for 3 epochs with a batch size of 16 using the HuggingFace "
        "Trainer API.",
        body_style),
    Paragraph(
        "Due to hardware constraints on the local training machine (Apple Silicon MPS), "
        "training was performed on a subsample of 10,000 examples (23% of the full training "
        "set) and 2,000 validation examples. This subsampling was a deliberate tradeoff to "
        "complete training within a feasible time window (~100 minutes). The same optimal "
        "threshold of 0.3 identified by the baseline sweep was applied to BERT predictions, "
        "as the threshold reflects a dataset-level property — both models output conservative "
        "sigmoid probabilities for rare labels under class imbalance — rather than a "
        "model-specific one.",
        body_style),
    tbl(
        [
            ["", "Baseline", "BERT"],
            ["Model",             "TF-IDF + Logistic Regression", "bert-base-uncased"],
            ["Training examples", "43,410 (full dataset)",        "10,000 (subsampled, 23%)"],
            ["Validation examples","5,426 (full)",                "2,000 (subsampled)"],
            ["Threshold",         "0.3 (tuned via sweep)",        "0.3 (same as baseline)"],
            ["Loss function",     "Log loss (per label)",         "BCEWithLogitsLoss"],
            ["Epochs",            "N/A",                          "3"],
            ["Training time",     "~2 minutes",                   "~100 minutes"],
        ],
        [4*cm, 7*cm, 5*cm]
    ),
    Spacer(1, 0.3*cm),
]

# ── 4. Evaluation Methodology ────────────────────────────────
story += [
    Paragraph("4. Evaluation Methodology", h1_style),
    Paragraph(
        "All models are evaluated on the same held-out test set of 5,427 examples. "
        "Four metrics are reported:",
        body_style),
    Paragraph("• <b>Micro F1</b> — harmonic mean of precision and recall, weighted by label frequency. "
              "Favors performance on common labels.", bullet_style),
    Paragraph("• <b>Macro F1</b> — unweighted average F1 across all 28 labels. "
              "Treats rare and common labels equally.", bullet_style),
    Paragraph("• <b>Precision</b> — fraction of predicted labels that are correct.", bullet_style),
    Paragraph("• <b>Recall</b> — fraction of true labels that are correctly predicted.", bullet_style),
    Paragraph(
        "Per-label F1 is additionally computed for all 28 emotion categories to identify "
        "which emotions are easiest and hardest for each model. All metrics are computed "
        "using scikit-learn's f1_score, precision_score, and recall_score with "
        "zero_division=0 to handle labels with no predictions gracefully.",
        body_style),
]

# ── 5. Results ───────────────────────────────────────────────
story += [
    Paragraph("5. Results", h1_style),
    Paragraph("5.1 Overall Metrics", h2_style),
    tbl(
        [
            ["Metric",    "Baseline", "BERT",  "Difference", "Winner"],
            ["Micro F1",  "0.5287",   "0.5586", "+0.0299",   "BERT"],
            ["Macro F1",  "0.3268",   "0.3025", "-0.0243",   "Baseline"],
            ["Precision", "0.5899",   "0.5880", "-0.0019",   "Baseline"],
            ["Recall",    "0.4791",   "0.5320", "+0.0529",   "BERT"],
        ],
        [4*cm, 3*cm, 3*cm, 3*cm, 3*cm]
    ),
    Spacer(1, 0.3*cm),
    Paragraph(
        "BERT outperforms the baseline on micro F1 (+0.030) and recall (+0.053), indicating "
        "it identifies more true emotion labels overall. The baseline achieves higher macro F1 "
        "(+0.024), reflecting stronger and more consistent performance across rare emotion "
        "categories. This is directly attributable to the training data disparity — the baseline "
        "trained on 43,410 examples while BERT used only 10,000, meaning BERT saw significantly "
        "fewer instances of rare emotions like grief, pride, and relief during training.",
        body_style),
    Paragraph(
        "Precision is essentially equal between the two models (0.5899 vs 0.5880), suggesting "
        "that when both models do make a prediction they are similarly accurate. The primary "
        "differentiator is recall — BERT's pretrained language representations allow it to "
        "identify more true labels even from limited training data.",
        body_style),
    Paragraph("5.2 Impact of Subsampling on BERT Performance", h2_style),
    Paragraph(
        "The decision to subsample BERT's training data to 10,000 examples was driven by "
        "hardware constraints and has measurable consequences on the results. Under full "
        "dataset training (43,410 examples), we would expect BERT's macro F1 to improve "
        "substantially — rare emotions like grief (77 training examples) and pride (111 "
        "training examples) would each appear fewer than 3 times in BERT's 10,000-example "
        "subsample on average, making it nearly impossible for the model to learn reliable "
        "representations for them. The baseline, having seen all instances of every label, "
        "maintains a structural advantage for rare categories.",
        body_style),
    Paragraph(
        "Despite this disadvantage, BERT's superior micro F1 and recall demonstrate that "
        "pretrained transformer representations are inherently more powerful for this task, "
        "and that the performance gap would likely widen significantly in BERT's favor "
        "with full dataset training.",
        body_style),
    Paragraph("5.3 Per-label F1 Breakdown", h2_style),
    tbl(
        [
            ["Label",         "Baseline F1", "BERT F1", "Delta",  "Observation"],
            ["gratitude",     "0.9142",      "0.9162",  "+0.002", "Both models excel — strong lexical signal"],
            ["amusement",     "0.7343",      "0.8163",  "+0.082", "BERT benefits from contextual understanding"],
            ["love",          "0.7328",      "0.8068",  "+0.074", "BERT captures affective context better"],
            ["neutral",       "0.6469",      "0.6683",  "+0.021", "Slight BERT advantage"],
            ["admiration",    "0.5950",      "0.6648",  "+0.070", "BERT stronger on praise language"],
            ["caring",        "0.2222",      "0.0000",  "-0.222", "BERT fails — ambiguous, rare"],
            ["fear",          "0.4466",      "0.0000",  "-0.447", "BERT fails — confused with surprise"],
            ["embarrassment", "0.1463",      "0.0000",  "-0.146", "BERT fails — insufficient training examples"],
            ["desire",        "0.2545",      "0.0000",  "-0.255", "BERT fails — subtle linguistic signal"],
            ["pride",         "0.1111",      "0.0000",  "-0.111", "BERT fails — extremely rare label"],
        ],
        [3.5*cm, 2.5*cm, 2.5*cm, 2*cm, 5.5*cm]
    ),
    Spacer(1, 0.3*cm),
    Paragraph(
        "BERT consistently outperforms the baseline on emotionally distinct, high-frequency "
        "labels where pretrained contextual representations provide a clear advantage. However, "
        "BERT scores exactly 0.0 on five labels even at threshold 0.3, indicating that the "
        "model never predicts these categories at all. This is a direct consequence of "
        "subsampling — with only 10,000 training examples, rare emotions appear so infrequently "
        "that BERT's classifier head never learns a reliable decision boundary for them.",
        body_style),
]

# ── 6. Error Analysis ────────────────────────────────────────
story += [
    Paragraph("6. Error Analysis", h1_style),
    Paragraph(
        "We examined BERT's misclassified examples on the test set and identified four "
        "systematic error patterns. These patterns reveal the nature of BERT's failures "
        "beyond what aggregate metrics can show.",
        body_style),
    Paragraph("Pattern 1 — Semantically close confusion", h2_style),
    Paragraph(
        "BERT confuses emotions that are semantically adjacent and share overlapping "
        "linguistic signals. 'Kings fan here, good luck to you guys! Will be an interesting "
        "game to watch!' was labelled excitement but predicted as optimism — both emotions "
        "involve positive anticipation and are difficult to distinguish without broader context. "
        "Similarly, 'I've also heard that intriguing but also kinda scary' was labelled fear "
        "but predicted as surprise, as both involve an element of the unexpected.",
        body_style),
    Paragraph("Pattern 2 — Keyword-driven misprediction", h2_style),
    Paragraph(
        "BERT sometimes latches onto a salient keyword while ignoring the broader emotional "
        "context of the sentence. 'I'm really sorry about your situation :( Although I love "
        "the names Sapphira, Cirilla, and Scarlett!' was labelled sadness but predicted as "
        "love — BERT focused on the word 'love' while missing the apologetic and empathetic "
        "tone established by 'I'm really sorry' and the sad emoji.",
        body_style),
    Paragraph("Pattern 3 — Missed secondary label in multi-label examples", h2_style),
    Paragraph(
        "In multi-label examples, BERT often correctly identifies the dominant emotion but "
        "misses the subtler secondary label. 'It's great that you're a recovering addict... "
        "Have you ever tried DMT?' carries both admiration and curiosity — BERT correctly "
        "predicted admiration but missed curiosity, whose signal is weaker and depends on "
        "recognizing the question as genuine inquiry rather than rhetoric.",
        body_style),
    Paragraph("Pattern 4 — Rare emotion missed entirely", h2_style),
    Paragraph(
        "'I wish I could afford it' expresses desire but BERT predicted nothing — an empty "
        "prediction even at threshold 0.3. Desire is one of the five labels where BERT scores "
        "0.0 F1 across the entire test set. This reflects the subsampling problem directly: "
        "desire appears only ~255 times in the full training set, meaning BERT's 10,000-example "
        "subsample contained approximately 59 desire examples — far too few to learn a "
        "reliable classifier.",
        body_style),
]

# ── 7. Discussion ────────────────────────────────────────────
story += [
    Paragraph("7. Discussion", h1_style),
    Paragraph(
        "The results reveal an important nuance: model architecture alone does not determine "
        "performance on multi-label emotion classification. BERT's micro F1 advantage "
        "(+0.030) over a much simpler TF-IDF baseline, achieved with only 23% of the "
        "training data, is a strong argument for the power of pretrained transformer "
        "representations. However, the baseline's macro F1 advantage demonstrates that "
        "training data coverage matters enormously for rare label categories.",
        body_style),
    Paragraph(
        "The threshold of 0.3 proved optimal for both models, which is consistent with "
        "findings from the original GoEmotions paper (Demszky et al., 2020) where lower "
        "thresholds are recommended due to class imbalance. Under sigmoid-based multi-label "
        "classification, both models are naturally conservative for rare labels — outputting "
        "probabilities well below 0.5 for categories they are uncertain about. A threshold "
        "of 0.5 would cause severe recall degradation for both models on this dataset.",
        body_style),
    Paragraph("Limitations", h2_style),
    Paragraph("• BERT trained on 10,000 examples (23% of full dataset) due to hardware constraints, "
              "directly impacting macro F1 and per-label performance on rare emotions.", bullet_style),
    Paragraph("• Threshold sweep limited to {0.3, 0.4, 0.5} — a finer sweep (e.g. 0.1 to 0.9 "
              "in steps of 0.05) may yield further improvements.", bullet_style),
    Paragraph("• No class weighting or data augmentation applied to address label imbalance.", bullet_style),
    Paragraph("• Error analysis focused on BERT only — a parallel analysis of baseline errors "
              "would provide a more complete picture.", bullet_style),
    Paragraph("• BERT evaluated on the full test set (5,427 examples) despite training on a "
              "subsample, which may underestimate performance on common labels relative to a "
              "fully trained model.", bullet_style),
]

# ── 8. Conclusion ────────────────────────────────────────────
story += [
    Paragraph("8. Conclusion", h1_style),
    Paragraph(
        "This project compared a TF-IDF + Logistic Regression baseline against a fine-tuned "
        "BERT model on the GoEmotions 28-label multi-label emotion classification task. BERT "
        "achieves higher micro F1 (0.5586 vs 0.5287) and recall (0.5320 vs 0.4791) despite "
        "training on only 23% of the data used by the baseline. The baseline achieves higher "
        "macro F1 (0.3268 vs 0.3025), reflecting stronger performance on rare emotion "
        "categories due to full dataset training.",
        body_style),
    Paragraph(
        "Error analysis reveals four systematic failure patterns in BERT: semantically close "
        "confusion, keyword-driven misprediction, missed secondary labels in multi-label "
        "examples, and complete failure on rare emotion categories. These patterns point to "
        "three concrete next steps: full dataset training for BERT, class-weighted loss to "
        "address label imbalance, and a finer threshold sweep. With these improvements, "
        "BERT's performance advantage over the baseline would likely be substantially larger.",
        body_style),
]

# ── References ───────────────────────────────────────────────
story += [
    PageBreak(),
    Paragraph("References", h1_style),
    Paragraph(
        "Demszky, D., Movshovitz-Attias, D., Ko, J., Cowen, A., Nemade, G., &amp; Ravi, S. (2020). "
        "GoEmotions: A dataset of fine-grained emotions. "
        "<i>Proceedings of the 58th Annual Meeting of the ACL</i>, pp. 4040–4054.",
        body_style),
    Paragraph(
        "Devlin, J., Chang, M. W., Lee, K., &amp; Toutanova, K. (2019). "
        "BERT: Pre-training of deep bidirectional transformers for language understanding. "
        "<i>NAACL-HLT 2019</i>, pp. 4171–4186.",
        body_style),
    Paragraph(
        "Wolf, T., Debut, L., Sanh, V., Chaumond, J., Delangue, C., Moi, A., ... &amp; Rush, A. M. (2020). "
        "HuggingFace Transformers: State-of-the-art natural language processing. "
        "<i>EMNLP 2020 (Systems Demonstrations)</i>, pp. 38–45.",
        body_style),
    Paragraph(
        "Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B., Grisel, O., ... &amp; Duchesnay, E. (2011). "
        "Scikit-learn: Machine learning in Python. "
        "<i>Journal of Machine Learning Research</i>, 12, pp. 2825–2830.",
        body_style),
    Paragraph(
        "Loshchilov, I., &amp; Hutter, F. (2019). Decoupled weight decay regularization. "
        "<i>International Conference on Learning Representations (ICLR 2019)</i>.",
        body_style),
    Paragraph(
        "Lhoest, Q., Villanova del Moral, A., Jernite, Y., Thakur, A., von Platen, P., Patil, S., ... &amp; Stretch, E. (2021). "
        "Datasets: A community library for natural language processing. "
        "<i>EMNLP 2021 (Systems Demonstrations)</i>, pp. 175–184.",
        body_style),
    Paragraph(
        "Zhang, M. L., &amp; Zhou, Z. H. (2014). A review on multi-label learning algorithms. "
        "<i>IEEE Transactions on Knowledge and Data Engineering</i>, 26(8), pp. 1819–1837.",
        body_style),
    Paragraph(
        "Manning, C. D., Raghavan, P., &amp; Schütze, H. (2008). "
        "<i>Introduction to Information Retrieval</i>. Cambridge University Press.",
        body_style),
]

# ── Build ────────────────────────────────────────────────────
doc.build(story)
print(f"\n✅ PDF saved: {PDF_NAME}")
print(f"📁 Location: {os.path.abspath(PDF_NAME)}")


✅ PDF saved: /content/GroupProject_GoEmotions_EmotionClassification_Report.pdf
📁 Location: /content/GroupProject_GoEmotions_EmotionClassification_Report.pdf
